# Create JSMF (James S. McDonnell Foundation) Awards

McDonnell Foundation grants scraped from jsmf.org/grants (FacetWP). ~1,161 grants
1987-2025 (cognitive/brain science + complex systems legacy + recent civic grants).

**Prerequisites:** Run `scripts/local/jsmf_to_s3.py` first.
**Data source:** https://www.jsmf.org/grants/
**S3:** `s3a://openalex-ingest/awards/jsmf/jsmf_grants.parquet`
**Funder:** funder_id 4320306183. Currency USD. Institution/grantee-level
(recipient org); per-PI data is detail-page-gated (Crawl-delay 10) -> later enrichment.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.jsmf_raw
USING delta
AS SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/jsmf/jsmf_grants.parquet`;


In [ ]:
%sql
-- Check row count (should be ~1161)
SELECT COUNT(*) as total_projects FROM openalex.awards.jsmf_raw;

In [ ]:
%sql
-- Inspect column names
DESCRIBE openalex.awards.jsmf_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.jsmf_raw LIMIT 5;

## Step 2: Create JSMF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.jsmf_awards
USING delta
AS
WITH
jsmf_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306183  -- James S. McDonnell Foundation
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        'USD' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        CAST(NULL AS STRING) as funder_scheme,
        'jsmf' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd') as start_date,
        CAST(NULL AS DATE) as end_date,
        CASE WHEN TRY_CAST(g.start_year AS INT) BETWEEN 1980 AND 2027 THEN TRY_CAST(g.start_year AS INT) ELSE NULL END as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.recipient IS NOT NULL THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.recipient as name,
                        CAST(NULL AS STRING) as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.jsmf_raw g
    CROSS JOIN jsmf_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'jsmf' AND priority = 245;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    245 as priority
FROM openalex.awards.jsmf_awards;

## Verification Queries

In [ ]:
%sql
-- Check row count (should be ~1161)
SELECT COUNT(*) as total_awards FROM openalex.awards.jsmf_awards;

In [ ]:
%sql
-- Sample the data
SELECT 
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    end_year
FROM openalex.awards.jsmf_awards 
LIMIT 10;

In [ ]:
%sql
-- Check funder distribution (should all be JSMF)
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.jsmf_awards
GROUP BY funder.display_name
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check program/scheme distribution
SELECT 
    funder_scheme,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000000, 2) as funding_billion_czk
FROM openalex.awards.jsmf_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check year distribution
SELECT 
    start_year,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000, 1) as funding_millions_czk
FROM openalex.awards.jsmf_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 20;

In [ ]:
%sql
-- Check data completeness
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_institution,
    COUNT(lead_investigator.affiliation.ids) as has_ror,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_with_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(SUM(amount)/1000000000, 2) as total_funding_billion_czk
FROM openalex.awards.jsmf_awards;

In [ ]:
%sql
-- Check top institutions
SELECT 
    lead_investigator.affiliation.name as institution,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000000, 2) as funding_billion_czk
FROM openalex.awards.jsmf_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name
ORDER BY cnt DESC
LIMIT 20;

In [ ]:
%sql
-- Verify data was inserted into openalex_awards_raw
SELECT COUNT(*) as jsmf_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'jsmf' AND priority = 245;